# Hybrid Pipeline on the THS 2017 Trips File

Rebuilds the full fusion chain — superzone and GS empirical-Bayes hybrids, correction-factor TAZ matrices, trips versions, and the 119-TAZ sub-matrices — with the **THS 2017 trips file** as the survey source (the primary survey basis, superseding the activities-file extraction).

Survey side: trips from `placeno` order per `PerID3` × `SurveyDay`, departure hour = origin row's `Dep_h` ∈ {6,7,8}, weight `new_wf`; `actTaz` (2636) → `TAZ_1250` → study TAZs by proportional cellular allocation; both ends inside the study area. Methodology is otherwise identical to the activities-based pipeline: $P^* = \lambda_A P^{survey} + (1-\lambda_A) P^{cell}$ with $\lambda_A = n_A/(n_A+k)$, $k$ by cross-day validation (JSD); then $R_{AB} = P^*(B|A)/P^{cell}(B|A)$, $\tilde C_{ij} = C_{ij}R_{AB}$, rows normalized.

In [1]:
import numpy as np
import pandas as pd
import os

# ---- survey trips at 1250 level ----
df = pd.read_excel('Input/trips_ths_2017.xlsx')
d = df.sort_values(['PerID3', 'SurveyDay', 'placeno']).copy()
g = d.groupby(['PerID3', 'SurveyDay'])
d['origin'] = g['actTaz'].shift(1)
d['trip_dep_h'] = g['Dep_h'].shift(1)
trips = d.dropna(subset=['origin', 'actTaz', 'trip_dep_h'])
trips = trips[trips['trip_dep_h'].isin([6, 7, 8])]

k26 = pd.read_excel('Input/TAZ_2636_Keys.xlsx').drop_duplicates('TAZ_2636')
map_2636_1250 = k26.set_index('TAZ_2636')['TAZ_1250']
trips = trips.assign(o1250=trips['origin'].map(map_2636_1250), d1250=trips['actTaz'].map(map_2636_1250))

# ---- cellular at study TAZ + allocation matrices ----
keys_raw = pd.read_csv('Input/Matrices/1270_02_09_2021_TAZ_North_keys.csv', encoding='windows-1255')
kn = keys_raw[['TAZ_1270', 'TAZ_NUMBER']].dropna(subset=['TAZ_NUMBER'])
kn['TAZ_NUMBER'] = kn['TAZ_NUMBER'].astype(int)
kd = kn.drop_duplicates('TAZ_NUMBER')
cellular = pd.read_csv('Input/Matrices/AvgDayHourlyTrips201819_1270_weekday_v1.csv')[['fromZone', 'ToZone', 'h6', 'h7', 'h8']]
c = cellular.merge(kd.rename(columns={'TAZ_1270': 'fromZone', 'TAZ_NUMBER': 'fromTaz'}), on='fromZone')
c = c.merge(kd.rename(columns={'TAZ_1270': 'ToZone', 'TAZ_NUMBER': 'ToTaz'}), on='ToZone')
cell_taz = c.groupby(['fromTaz', 'ToTaz'])[['h6', 'h7', 'h8']].sum().sum(axis=1).unstack().fillna(0)
TAZ = cell_taz.index
out_vol, in_vol = cell_taz.sum(axis=1), cell_taz.sum(axis=0)

north_1250 = sorted(kd[kd['TAZ_NUMBER'].isin(TAZ)]['TAZ_1270'].unique())
z_idx = {z: i for i, z in enumerate(north_1250)}
children = kd[kd['TAZ_NUMBER'].isin(TAZ)].groupby('TAZ_1270')['TAZ_NUMBER'].apply(list)
taz_pos = {t: j for j, t in enumerate(TAZ)}

def alloc_matrix(vol):
    S = np.zeros((len(north_1250), len(TAZ)))
    for z, kids in children.items():
        v = np.array([vol[t] for t in kids], float)
        s = v / v.sum() if v.sum() > 0 else np.full(len(kids), 1 / len(kids))
        for t, sh in zip(kids, s):
            S[z_idx[z], taz_pos[t]] = sh
    return S

S_o, S_d = alloc_matrix(out_vol), alloc_matrix(in_vol)
in_study = trips['o1250'].isin(z_idx) & trips['d1250'].isin(z_idx)
tm = trips[in_study]
print(f"study-area AM trips: day 1 = {(tm['SurveyDay']==1).sum():,}, day 2 = {(tm['SurveyDay']==2).sum():,} sampled")

def taz_day(day):
    td = tm[tm['SurveyDay'] == day]
    m = td.groupby(['o1250', 'd1250'])['new_wf'].sum().unstack().fillna(0)
    m = m.reindex(index=north_1250, columns=north_1250, fill_value=0)
    return S_o.T @ m.values @ S_d

W1, W2 = taz_day(1), taz_day(2)
W_avg = (W1 + W2) / 2
print(f"avg-weekday study-area trips: {W_avg.sum():,.0f}")

study-area AM trips: day 1 = 13,476, day 2 = 13,387 sampled
avg-weekday study-area trips: 2,068,158


## Zone-system machinery and cross-day validation of k (SZ and GS)

In [2]:
taz_to_sz = keys_raw[['TAZ_NUMBER', 'SZ_NEW']].dropna().astype(int).drop_duplicates('TAZ_NUMBER').set_index('TAZ_NUMBER')['SZ_NEW']
taz_to_gs = pd.read_csv('Input/TAZ_GSnew.csv').set_index('TAZ')['GS']

def indicator(mapping):
    zones = sorted({mapping[t] for t in TAZ if t in mapping.index or t in mapping})
    pos = {z: i for i, z in enumerate(zones)}
    M = np.zeros((len(TAZ), len(zones)))
    for j, t in enumerate(TAZ):
        M[j, pos[mapping[t]]] = 1
    return zones, M

SZ, M_sz = indicator(taz_to_sz)
GS, M_gs = indicator(taz_to_gs)

# unweighted observation counts per origin zone (per day and pooled), via dominant zone of each 1250-zone
def dominant_map(mapping):
    dom = {}
    for z, kids in children.items():
        vals = pd.Series([mapping[t] for t in kids])
        vols = pd.Series([out_vol[t] for t in kids])
        dom[z] = vals.iloc[int(np.argmax(vols.groupby(vals).sum().reindex(vals).values))] if len(set(vals)) > 1 else vals.iloc[0]
        dom[z] = vals.groupby(vals).apply(lambda s: 0).index[0] if False else dom[z]
    return dom

dom_sz = {z: pd.Series({t: taz_to_sz[t] for t in kids}).groupby(lambda i: taz_to_sz[i]).count().idxmax() if len({taz_to_sz[t] for t in kids}) > 1 else taz_to_sz[kids[0]] for z, kids in children.items()}
dom_gs = {z: pd.Series({t: taz_to_gs[t] for t in kids}).groupby(lambda i: taz_to_gs[i]).count().idxmax() if len({taz_to_gs[t] for t in kids}) > 1 else taz_to_gs[kids[0]] for z, kids in children.items()}

def counts_by(dom, zones):
    out = {}
    for day in (1, 2):
        n = tm[tm['SurveyDay'] == day].groupby(tm['o1250'].map(dom)).size()
        out[day] = n.reindex(zones).fillna(0)
    return out

def agg_mat(W, M):
    A = M.T @ W @ M
    rs = A.sum(axis=1, keepdims=True)
    return A, np.where(rs > 0, A / rs, 0)

cell_sz_raw, P_cell_sz = agg_mat(cell_taz.values, M_sz)
cell_gs_raw, P_cell_gs = agg_mat(cell_taz.values, M_gs)

def jsd(p, q):
    if p.sum() <= 0 or q.sum() <= 0: return np.nan
    p, q = p / p.sum(), q / q.sum(); m = (p + q) / 2
    kl = lambda a, b: np.sum(a[a > 0] * np.log2(a[a > 0] / b[a > 0]))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)

def cv_level(M, P_cell, dom, zones):
    _, P1 = agg_mat(W1, M)
    _, P2 = agg_mat(W2, M)
    n = counts_by(dom, zones)
    rows = {}
    for k in [0, 1, 2, 5, 10, 20, 50, 100, 200, 500, np.inf]:
        def blend(P, nn):
            lam = np.where(nn.values > 0, nn.values / (nn.values + k), 0.0) if np.isfinite(k) else np.zeros(len(nn))
            return P * lam[:, None] + P_cell * (1 - lam[:, None])
        j1 = np.nanmean([jsd(blend(P1, n[1])[i], P2[i]) for i in range(len(zones))])
        j2 = np.nanmean([jsd(blend(P2, n[2])[i], P1[i]) for i in range(len(zones))])
        rows[k] = (j1 + j2) / 2
    return pd.Series(rows), P1, P2, n

cv_sz, P1_sz, P2_sz, n_sz = cv_level(M_sz, P_cell_sz, dom_sz, SZ)
cv_gs, P1_gs, P2_gs, n_gs = cv_level(M_gs, P_cell_gs, dom_gs, GS)
k_sz, k_gs = cv_sz.idxmin(), cv_gs.idxmin()
print("CV (mean row JSD) — SZ level:"); print(cv_sz.round(5).to_string())
print(f"k* (SZ) = {k_sz:g} | k* (GS) = {k_gs:g}")

CV (mean row JSD) — SZ level:
0.0      0.01951
1.0      0.01930
2.0      0.01919
5.0      0.01907
10.0     0.01923
20.0     0.02024
50.0     0.02518
100.0    0.03440
200.0    0.05055
500.0    0.08175
inf      0.17106
k* (SZ) = 5 | k* (GS) = 5


## Final hybrids (SZ and GS), trips versions

In [3]:
os.makedirs('Output/ths2017/study_taz', exist_ok=True)

def build_hybrid(M, P_cell, n, zones, k_star, tag):
    W_avg_z, P_avg = agg_mat(W_avg, M)
    n_pool = (n[1] + n[2]).values
    k_eff = max(float(k_star), 1e-9)
    lam = np.where(n_pool > 0, n_pool / (n_pool + k_eff), 0.0)
    P_h = P_avg * lam[:, None] + P_cell * (1 - lam[:, None])
    vol = W_avg_z.sum(axis=1)
    T_h = P_h * vol[:, None]
    pd.DataFrame(P_h, index=zones, columns=zones).to_csv(f'Output/ths2017/study_taz/hybrid_{tag}_prob.csv')
    pd.DataFrame(T_h, index=zones, columns=zones).to_csv(f'Output/ths2017/study_taz/hybrid_{tag}_trips.csv')
    lam_tbl = pd.DataFrame({'n_pooled': n_pool.astype(int), 'lambda': lam}, index=zones)
    lam_tbl.index.name = f'origin_{tag}'
    lam_tbl.to_csv(f'Output/ths2017/study_taz/hybrid_{tag}_lambda.csv')
    print(f"hybrid_{tag}: k* = {k_star:g}, lambda {lam[n_pool>0].min():.3f}-{lam.max():.4f}, trips {T_h.sum():,.0f}")
    return P_h, vol

P_h_sz, vol_sz = build_hybrid(M_sz, P_cell_sz, n_sz, SZ, k_sz, 'sz')
P_h_gs, vol_gs = build_hybrid(M_gs, P_cell_gs, n_gs, GS, k_gs, 'gs')
pd.concat([cv_sz.rename('SZ'), cv_gs.rename('GS')], axis=1).to_csv('Output/ths2017/study_taz/hybrid_cv_results.csv')

hybrid_sz: k* = 5, lambda 0.962-0.9971, trips 2,068,158
hybrid_gs: k* = 5, lambda 0.706-0.9994, trips 2,068,158


## Correction factors → TAZ-level hybrid matrices (SZ- and GS-calibrated)

In [4]:
col_sz = np.argmax(M_sz, axis=1)
col_gs = np.argmax(M_gs, axis=1)

def taz_hybrid(P_h, P_cell_agg, col, zones, vol, M, tag):
    R = np.where(P_cell_agg > 0, P_h / np.where(P_cell_agg == 0, 1, P_cell_agg), 1.0)
    Ct = cell_taz.values * R[col][:, col]
    rs = Ct.sum(axis=1, keepdims=True)
    P_taz = np.where(rs > 0, Ct / rs, 0)
    # origin volumes: zone survey departures split within-zone by cellular outflow shares
    share = np.zeros(len(TAZ))
    for zi in range(len(zones)):
        m = col == zi
        tot = out_vol.values[m].sum()
        share[m] = out_vol.values[m] / tot if tot > 0 else 1.0 / m.sum()
    T_origin = vol[col] * share
    T_taz = P_taz * T_origin[:, None]
    check = M.T @ T_taz.sum(axis=1)
    assert np.allclose(check, vol), "zone origin totals must match survey departures"
    pd.DataFrame(R, index=zones, columns=zones).to_csv(f'Output/ths2017/study_taz/{tag}_correction_factors.csv', float_format='%.6g')
    pd.DataFrame(P_taz, index=TAZ, columns=TAZ).to_csv(f'Output/ths2017/study_taz/hybrid_taz_prob{"" if tag=="sz" else "_gs"}.csv', float_format='%.6g')
    Tdf = pd.DataFrame(T_taz, index=TAZ, columns=TAZ)
    Tdf.to_csv(f'Output/ths2017/study_taz/hybrid_taz_trips{"" if tag=="sz" else "_gs"}.csv', float_format='%.6g')
    diag = np.diag(R); off = R[~np.eye(len(zones), dtype=bool)]
    print(f"{tag.upper()}-calibrated TAZ hybrid: {T_taz.sum():,.0f} trips | R diag median {np.median(diag):.2f}, off-diag median {np.median(off):.3f}")
    return Tdf

T_taz_sz = taz_hybrid(P_h_sz, P_cell_sz, col_sz, SZ, vol_sz, M_sz, 'sz')
T_taz_gs = taz_hybrid(P_h_gs, P_cell_gs, col_gs, GS, vol_gs, M_gs, 'gs')

SZ-calibrated TAZ hybrid: 2,068,158 trips | R diag median 2.04, off-diag median 0.180


GS-calibrated TAZ hybrid: 2,068,158 trips | R diag median 1.79, off-diag median 0.169


## Sub-matrices — aggregated areas (`Input/Submatrix_tazs.xlsx`)

The sub-area is defined by the 205 TAZs in `Submatrix_tazs.xlsx`, grouped into 28 named aggregated areas (`AggAreaCode` / `AggAreaName`) with an `IsLRT_Corridor` flag (1 for the 119 corridor TAZs — identical to the previous sub-matrix list — plus 86 non-corridor TAZs). Each matrix is restricted to trips with **both ends among the listed TAZs** and aggregated to the 28 areas.

In [5]:
sub_key = pd.read_excel('Input/Submatrix_tazs.xlsx')
taz_to_area = sub_key.set_index('TAZ')['AggAreaCode']
AREAS = sorted(sub_key['AggAreaCode'].unique())
corridor_tazs = set(sub_key.loc[sub_key['IsLRT_Corridor'] == 1, 'TAZ'])

legend = (sub_key.groupby(['AggAreaCode', 'AggAreaName'])
          .agg(n_TAZs=('TAZ', 'size'), corridor_TAZs=('IsLRT_Corridor', 'sum'))
          .reset_index())
legend['IsLRT_Corridor'] = np.where(legend['corridor_TAZs'] == legend['n_TAZs'], 1,
                            np.where(legend['corridor_TAZs'] == 0, 0, -1))  # -1 = mixed
os.makedirs('Output/ths2017/study_taz/submatrices', exist_ok=True)
legend.to_csv('Output/ths2017/study_taz/submatrices/area_legend.csv', index=False)
print(f"sub-area: {len(sub_key)} TAZs -> {len(AREAS)} aggregated areas "
      f"({len(corridor_tazs)} corridor TAZs; {int((legend['IsLRT_Corridor'] == -1).sum())} areas mixed)")

def to_area_matrix(m):
    m = m.copy()
    m.index.name, m.columns.name = 'o', 'd'
    long = m.stack().reset_index()
    long.columns = ['o', 'd', 'v']
    long['O'] = long['o'].map(taz_to_area)
    long['D'] = long['d'].map(taz_to_area)
    return (long.dropna(subset=['O', 'D']).groupby(['O', 'D'])['v'].sum()
            .unstack().reindex(index=AREAS, columns=AREAS, fill_value=0).fillna(0))

sources = {f'matrix_avg_{m}_area.csv': pd.read_csv(f'Output/ths2017/study_taz/matrix_avg_{m}_taz.csv', index_col=0)
           for m in ['ALL', 'CAR', 'TRANSIT', 'RAIL', 'OTHER']}
sources['hybrid_taz_trips_area.csv'] = T_taz_sz

for name, m in sources.items():
    m.index = m.index.astype(float)
    m.columns = m.columns.astype(float)
    area = to_area_matrix(m)
    area.index.name = 'AggAreaCode'
    area.to_csv(f'Output/ths2017/study_taz/submatrices/{name}', float_format='%.6g')
    corr_mask_o = [t for t in m.index if t in corridor_tazs]
    corr_v = m.loc[[t for t in m.index if t in corridor_tazs], [t for t in m.columns if t in corridor_tazs]].values.sum()
    print(f"{name}: {area.values.sum():>10,.0f} trips in the sub-area | {corr_v:>10,.0f} with both ends in the LRT corridor")

sub-area: 205 TAZs -> 28 aggregated areas (119 corridor TAZs; 2 areas mixed)


matrix_avg_ALL_area.csv:    276,977 trips in the sub-area |    110,641 with both ends in the LRT corridor
matrix_avg_CAR_area.csv:    176,732 trips in the sub-area |     66,638 with both ends in the LRT corridor
matrix_avg_TRANSIT_area.csv:     26,247 trips in the sub-area |      9,459 with both ends in the LRT corridor
matrix_avg_RAIL_area.csv:        131 trips in the sub-area |          0 with both ends in the LRT corridor
matrix_avg_OTHER_area.csv:     73,866 trips in the sub-area |     34,545 with both ends in the LRT corridor
hybrid_taz_trips_area.csv:    273,320 trips in the sub-area |     94,610 with both ends in the LRT corridor


## Notes

- This pipeline supersedes the activities-based hybrid products (`Output/hybrid_*`) as the primary set; those remain in place as the historical version.
- All caveats carry over: the cellular replication mapping, the intra-zone survey/cellular divergence, cross-day validation's same-panel limitation, and — new here — within-1250-zone spatial detail on the survey side comes from cellular allocation shares, so at TAZ resolution the survey contributes pattern only down to the 1250-zone level.
- Observation counts n_A for λ use the dominant SZ/GS of each 1250-zone (17 of 400 zones span >1 superzone, 8 span >1 GS — negligible for counting).